In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os
import datetime


project_drive_path = "/content/drive/MyDrive/DermaVue Project"
project_local_path = "/content/DermaVue_Project"
model_path = f"{project_drive_path}/mobilenetv2_finetuned_augmentation.keras"

os.makedirs(project_local_path, exist_ok=True)

for filename in os.listdir(project_drive_path):

    full_source_path = os.path.join(project_drive_path, filename)
    full_target_path = os.path.join(project_local_path, filename)

    if os.path.isfile(full_source_path):
        shutil.copy(full_source_path, full_target_path)
        print(f"Copied: {filename}")
    else:
        print(f"Skipped directory: {filename}")

os.chdir(project_local_path)
print(f"\nCurrent working directory set to: {os.getcwd()}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied: mobilenetv2_finetuned_best.keras
Copied: skincare_products_clean.csv
Copied: mobilenetv2_finetuned_augmentation.keras
Copied: mobilenetv2_final_manual_thresh.keras
Copied: Recommender System Evaluation.ipynb
Copied: User Interface - Streamlit.ipynb

Current working directory set to: /content/DermaVue_Project


In [ ]:
from tensorflow.keras.models import load_model

project_drive_path = "/content/drive/MyDrive/DermaVue Project"
model_path = f"{project_drive_path}/mobilenetv2_finetuned_augmentation.keras"
model = load_model(model_path, compile=False)
print("Model loaded for inference (no macro_f1 needed)")

Model loaded for inference (no macro_f1 needed)


In [ ]:
import os
import shutil

source_folder = '/content/drive/MyDrive/DermaVue Project'
target_folder = '/content/DermaVue Project'

os.makedirs(target_folder, exist_ok=True)

for filename in os.listdir(source_folder):
    if filename.endswith('.gsheet'):
        print(f"Skipped Google Sheet link: {filename}")
        continue

    full_source_path = os.path.join(source_folder, filename)
    full_target_path = os.path.join(target_folder, filename)

    if os.path.isfile(full_source_path):
        try:
            shutil.copy(full_source_path, full_target_path)
            print(f"Copied: {filename}")
        except Exception as e:
            print(f"Failed to copy {filename}: {e}")

Copied: mobilenetv2_finetuned_best.keras
Copied: skincare_products_clean.csv
Copied: mobilenetv2_finetuned_augmentation.keras
Copied: mobilenetv2_final_manual_thresh.keras
Copied: Recommender System Evaluation.ipynb
Copied: User Interface - Streamlit.ipynb


In [ ]:
%%writefile style.css
:root {
    --bg-light: #EFF8FD;
    --card-primary: #DEF3FF;
    --card-secondary: #DDF1FF;
    --border-light: #CBDEEB;
    --text-dark: #333333;
    --text-muted: #787878;
}

body {
    background-color: var(--bg-light);
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color: var(--text-dark);
}

.title {
    font-size: 2.5em;
    font-weight: 600;
    color: var(--text-dark);
    text-align: center;
    letter-spacing: -0.5px;
    margin-top: 1rem;
    margin-bottom: 0.5rem;
}

.subtitle {
    font-size: 1.1em;
    color: var(--text-muted);
    font-style: normal;
    text-align: center;
    margin-top: 0;
    margin-bottom: 2rem;
}

button {
    background-color: var(--card-secondary);
    color: var(--text-dark);
    font-weight: 500;
    border-radius: 3px;
    padding: 0.6rem 1.2rem;
    border: 1px solid var(--border-light);
}


.stFileUploader,
.stTextInput,
.stNumberInput {
    border-radius: 4px;
    border: 1px solid var(--border-light);
}


h3 {
    font-size: 1.4em;
    font-weight: 600;
    color: var(--text-dark);
    letter-spacing: -0.2px;
}

Overwriting style.css


In [ ]:
%%writefile app.py
import streamlit as st
from PIL import Image
import numpy as np
import pandas as pd
import tensorflow as tf
import os
from tensorflow.keras.models import load_model
import datetime

if "form_done" not in st.session_state:
    st.session_state.form_done = False
if "preferences_done" not in st.session_state:
    st.session_state.preferences_done = False
if "uploaded_img" not in st.session_state:
    st.session_state.uploaded_img = None
if "show_results" not in st.session_state:
    st.session_state.show_results = False
if "price_min" not in st.session_state:
    st.session_state.price_min = 5.0
if "price_max" not in st.session_state:
    st.session_state.price_max = 50.0
if "preferred_types" not in st.session_state:
    st.session_state.preferred_types = []

# --- Utils ---
def preprocess_image(image):
    image = image.resize((224, 224))
    image_array = np.array(image) / 255.0
    if image_array.shape[-1] == 4:
        image_array = image_array[..., :3]
    image_array = np.expand_dims(image_array, axis=0)
    return image_array

def recommend_products(conditions, price_min, price_max, df, preferred_types=None):
    condition_to_ingredients_weighted = {
    "Acne": {
        "salicylic acid": 3,
        "benzoyl peroxide": 3,
        "retinoids": 3,
        "niacinamide": 3,
        "azelaic acid": 3,
        "adapalene": 1,
        "alpha-hydroxy acids": 1,
        "sulfur": 1,
        "tea tree oil": 1,
        "succinic acid": 1
    },
    "Dry Skin": {
        "hyaluronic acid": 3,
        "ceramides": 3,
        "niacinamide": 3,
        "glycerin": 3,
        "shea butter": 3,
        "squalane": 1,
        "vitamin e": 1,
        "dimethicone": 1,
        "petrolatum": 1
    },
    "Oily Skin": {
        "salicylic acid": 3,
        "niacinamide": 3,
        "clay": 3,
        "hyaluronic acid": 3,
        "retinol": 3,
        "tea tree oil": 1,
        "lactic acid": 1,
        "astringents": 1
    },
    "Dark Spots": {
        "hydroquinone": 3,
        "vitamin c": 3,
        "kojic acid": 3,
        "retinol": 3,
        "niacinamide": 3,
        "glycolic acid": 1,
        "azelaic acid": 1,
        "tranexamic acid": 1
    },
    "Wrinkles": {
        "hyaluronic acid": 3,
        "collagen": 3,
        "retinol": 3,
        "vitamin c": 3,
        "niacinamide": 3,
        "glycolic acid": 1,
        "bakuchiol": 1,
        "ferulic acid": 1,
        "peptides": 1,
        "polyglutamic acid": 1
    },
    "Eye bags": {
        "hyaluronic acid": 3,
        "adipoless": 3,
        "peptides": 3,
        "sabiwhite": 3,
        "syn-eye": 3,
        "caffeine": 1,
        "vitamin c": 1
    },
    "Pores": {
        "niacinamide": 3,
        "green tea": 3,
        "azelaic acid": 3,
        "salicylic acid": 3,
        "retinol": 3,
        "alpha-hydroxy acids": 1,
        "charcoal": 1,
        "kaolin clay": 1
    },
    "Normal Skin": {
        "ceramides": 3,
        "hyaluronic acid": 3,
        "niacinamide": 3,
        "squalane": 3,
        "lactic acid": 3
    },
    "Combination Skin": {
        "superoxide dismutase": 3,
        "lactic acid": 3,
        "hyaluronic acid": 3,
        "peptides": 3,
        "squalane": 3,
        "alpha-hydroxy acids": 1,
        "green clay": 1,
        "salicylic acid": 1,
        "glycolic acid": 1,
        "witch hazel": 1
    }
}

    def score_product_weighted(row):
        try:
            ingreds = [i.lower() for i in eval(row['clean_ingreds'])]
        except:
            ingreds = []

        score = 0
        for cond in conditions:
            weights = condition_to_ingredients_weighted.get(cond, {})
            for ing in ingreds:
                score += weights.get(ing, 0)

        type_bonus = 1 if preferred_types and row['product_type'] in preferred_types else 0
        return score + type_bonus

    df_filtered = df[
        (df['price_clean'] >= price_min) & (df['price_clean'] <= price_max)
    ].copy()

    df_filtered['match_score'] = df_filtered.apply(score_product_weighted, axis=1)
    results = df_filtered[df_filtered['match_score'] > 0]
    results = results.sort_values(by='match_score', ascending=False)

    return results.head(5).to_dict(orient='records')

project_drive_path = "/content/drive/MyDrive/DermaVue Project"
project_local_path = "/content/DermaVue_Project"
product_path = os.path.join(project_local_path, "skincare_products_clean.csv")
df_products = pd.read_csv(product_path)
df_products['price_clean'] = (
    df_products['price']
    .replace('[£€,]', '', regex=True)
    .str.strip()
    .astype(float))


model_path = os.path.join(project_drive_path, "mobilenetv2_finetuned_augmentation.keras")
def macro_f1(y_true, y_pred):
    return tf.constant(0.0)

model = load_model(model_path, compile=False)

if "form_done" not in st.session_state:
    st.session_state.form_done = False
if "preferences_done" not in st.session_state:
    st.session_state.preferences_done = False
if "price_min" not in st.session_state:
    st.session_state.price_min = 5.0
if "price_max" not in st.session_state:
    st.session_state.price_max = 50.0
if "preferred_types" not in st.session_state:
    st.session_state.preferred_types = []
if "uploaded_img" not in st.session_state:
    st.session_state.uploaded_img = None
if "show_results" not in st.session_state:
    st.session_state.show_results = False
if "just_saved_preferences" not in st.session_state:
    st.session_state.just_saved_preferences = False


with st.form("user_info"):
    st.markdown("### Tell us about yourself!")
    col1, col2 = st.columns(2)
    name = col1.text_input("Name")
    email = col2.text_input("Email")
    gender = col1.selectbox("Gender", ["Female", "Male", "Other"])
    min_dob = datetime.date(1960, 1, 1)
    max_dob = datetime.date(2010, 12, 31)

    dob = col2.date_input(
        "Date of Birth",
        min_value=min_dob,
        max_value=max_dob,
        # optional: set a sensible default, e.g. midpoint:
        value=datetime.date(1985, 1, 1)
    )
    country = st.text_input("Country")
    submitted = st.form_submit_button("Continue")

if submitted:
    st.session_state.form_done = True

if st.session_state.form_done and not st.session_state.preferences_done:
    st.markdown("### Step 1: Preferences")

    col1, col2 = st.columns(2)
    price_min = col1.number_input("Price from (€)", value=st.session_state.price_min)
    price_max = col2.number_input("Price to (€)", value=st.session_state.price_max)

    product_type_options = [
        "Moisturiser", "Serum", "Oil", "Mist", "Balm", "Mask", "Peel",
        "Eye Care", "Cleanser", "Toner", "Exfoliator", "Bath Salts", "Body Wash", "Bath Oil"
    ]
    preferred_types = st.multiselect(
        "Choose preferred product types (optional):",
        product_type_options,
        default=st.session_state.preferred_types
    )

    if st.button("Save preferences"):
        st.session_state.price_min = price_min
        st.session_state.price_max = price_max
        st.session_state.preferred_types = preferred_types
        st.session_state.preferences_done = True
        st.rerun()

if st.session_state.preferences_done and st.session_state.uploaded_img is None:
    st.markdown("### Step 2: Upload your image")
    uploaded = st.file_uploader("Upload your face image", type=["jpg", "jpeg", "png"])

    if uploaded:
        st.session_state.uploaded_img = uploaded
        st.image(Image.open(uploaded), caption="Uploaded Image", use_container_width=True)

if st.session_state.uploaded_img is not None and not st.session_state.show_results:
    if st.button("Get my targeted product recommendations!"):
        st.session_state.show_results = True
        st.rerun()

if st.session_state.show_results:
    image = Image.open(st.session_state.uploaded_img)
    input_tensor = preprocess_image(image)

    preds = model.predict(input_tensor)[0]

    class_labels = ['Acne', 'Dark Spots', 'Dry Skin', 'Eye bags', 'Normal Skin',
                    'Oily Skin', 'Pores', 'Wrinkles']
    detected_conditions = [label for label, prob in zip(class_labels, preds) if prob > 0.5]

    if "Dry Skin" in detected_conditions and "Oily Skin" in detected_conditions:
        detected_conditions = [c for c in detected_conditions if c not in ["Dry Skin", "Oily Skin"]]
        detected_conditions.append("Combination Skin")

    st.markdown("### Your Detected Skin Conditions:")
    st.write(", ".join(detected_conditions) if detected_conditions else "No specific conditions detected.")

    st.markdown("### Your Product Recommendations")
    recs = recommend_products(
        detected_conditions,
        st.session_state.price_min,
        st.session_state.price_max,
        df_products,
        st.session_state.preferred_types
    )

    if recs:
        for r in recs:
            st.markdown(f"**[{r['product_name']}]({r['product_url']})**")
            st.markdown(f"- Price: €{r['price_clean']:.2f}")
            st.markdown(f"- Type: {r['product_type']}")
            st.markdown("---")
    else:
        st.info("No matching products found.")

Overwriting app.py


In [ ]:
!pip install -q streamlit pyngrok

import subprocess
import time
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "2v0IhRxfGIszxomre0BsCxUuoaj_27QRnRytcoVDxjVMLYjF9"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

!pkill streamlit

print("Starting Streamlit app...")
streamlit_process = subprocess.Popen(["streamlit", "run", "app.py"])

time.sleep(5)

public_url = ngrok.connect(8501)
print("Public DermaVue App URL:", public_url)

Starting Streamlit app...
Public DermaVue App URL: NgrokTunnel: "https://1e1f-34-16-145-15.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
#from pyngrok import ngrok
#ngrok.kill()